In [1]:


import numpy as np
import pandas as pd
import seaborn as sns
import jax
import functools
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import rapids_singlecell as rsc
import flax.linen as nn
import optax
import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear
from cellflow.plotting import plot_condition_embedding
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca, reconstruct_pca
from cellflow.metrics import compute_r_squared, compute_e_distance

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [3]:
adata = cellflow.datasets.pbmc_cytokines()
adata.obs["condition"] = adata.obs.apply(lambda x: x["donor"] + "_" + x["cytokine"], axis=1)
adata.obs["is_control"] = adata.obs.apply(lambda x: True if x["cytokine"]=="PBS" else False, axis=1)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata_train = adata[(adata.obs["cytokine"]!="IL-15") | (adata.obs["donor"]=="Donor8")].copy()
adata_test = adata[((adata.obs["cytokine"]=="IL-15") & (adata.obs["donor"]!="Donor8")) | (adata.obs["cytokine"]=="PBS")].copy()
adata_train.n_obs, adata_test.n_obs

cfpp.centered_pca(adata_train, n_comps=100, method="rapids", keep_centered_data=False)
cfpp.project_pca(query_adata=adata_test, ref_adata=adata_train)

In [4]:
from memory_profiler import memory_usage

In [6]:

def run_cellflow():
    cf = CellFlow(adata_train, solver="otfm")
    cf.prepare_data(
        sample_rep = "X_pca",
        control_key = "is_control",
        perturbation_covariates = {"cytokine_treatment": ("cytokine",)},
        perturbation_covariate_reps = {"cytokine_treatment": "esm2_embeddings"},
        sample_covariates = ["donor"],
        sample_covariate_reps = {"donor": "donor_embeddings"},
        split_covariates = ["donor"],
        max_combination_length = 1,
        null_value = 0.0,
    )
    layers_before_pool = {
        "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
        "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
    }

    layers_after_pool = {
        "layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0,
    }

    match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)
    cf.prepare_model(
        condition_mode="deterministic",
        regularization=0.0,
        pooling="attention_token",
        pooling_kwargs={},
        layers_before_pool=layers_before_pool,
        layers_after_pool=layers_after_pool,
        condition_embedding_dim=256,
        cond_output_dropout=0.9,
        condition_encoder_kwargs={},
        pool_sample_covariates=True,
        time_freqs=1024,
        time_encoder_dims=[1024, 1024, 1024],
        time_encoder_dropout=0.0,
        hidden_dims=[2048, 2048, 2048],
        hidden_dropout=0.0,
        conditioning="concatenation",
        decoder_dims=[4096, 4096, 4096],
        vf_act_fn=nn.silu,
        vf_kwargs=None,
        probability_path={"constant_noise": 0.5},
        match_fn=match_fn,
        optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
        solver_kwargs={},
        layer_norm_before_concatenation=False,
        linear_projection_before_concatenation=False,
    )

    cf.train(
        num_iterations=10,
        batch_size=1024,
        callbacks=[],
        valid_freq=20_000,
    )


mem = memory_usage(run_cellflow, max_usage=True)
print(f"Peak memory: {mem:.2f} MiB")

[########################################] | 100% Completed | 103.01 ms
[########################################] | 100% Completed | 2.30 sms
[########################################] | 100% Completed | 209.29 ms


100%|██████████| 10/10 [00:11<00:00,  1.17s/it]


Peak memory: 236991.11 MiB


In [ ]:

def run_cellflow():
    cf = CellFlow(adata_train, solver="otfm")
    cf.prepare_data(
        sample_rep = "X_pca",
        control_key = "is_control",
        perturbation_covariates = {"cytokine_treatment": ("cytokine",)},
        perturbation_covariate_reps = {"cytokine_treatment": "esm2_embeddings"},
        sample_covariates = ["donor"],
        sample_covariate_reps = {"donor": "donor_embeddings"},
        split_covariates = ["donor"],
        max_combination_length = 1,
        null_value = 0.0,
    )
    layers_before_pool = {
        "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
        "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
    }

    layers_after_pool = {
        "layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0,
    }

    match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)
    cf.prepare_model(
        condition_mode="deterministic",
        regularization=0.0,
        pooling="attention_token",
        pooling_kwargs={},
        layers_before_pool=layers_before_pool,
        layers_after_pool=layers_after_pool,
        condition_embedding_dim=256,
        cond_output_dropout=0.9,
        condition_encoder_kwargs={},
        pool_sample_covariates=True,
        time_freqs=1024,
        time_encoder_dims=[1024, 1024, 1024],
        time_encoder_dropout=0.0,
        hidden_dims=[2048, 2048, 2048],
        hidden_dropout=0.0,
        conditioning="concatenation",
        decoder_dims=[4096, 4096, 4096],
        vf_act_fn=nn.silu,
        vf_kwargs=None,
        probability_path={"constant_noise": 0.5},
        match_fn=match_fn,
        optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
        solver_kwargs={},
        layer_norm_before_concatenation=False,
        linear_projection_before_concatenation=False,
    )

    cf.train(
        num_iterations=10,
        batch_size=1024,
        callbacks=[],
        valid_freq=20_000,
    )


mem = memory_usage(run_cellflow, max_usage=True)
print(f"Peak memory: {mem:.2f} MiB")

In [7]:
def run_cellflow1():
    cf = CellFlow(adata_train, solver="otfm")

In [8]:
mem = memory_usage(run_cellflow1, max_usage=True)
print(f"Peak memory: {mem:.2f} MiB")

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Peak memory: 163996.52 MiB


In [9]:
cf = CellFlow(adata_train, solver="otfm")

In [11]:
def run_cellflow2():
    
    cf.prepare_data(
        sample_rep = "X_pca",
        control_key = "is_control",
        perturbation_covariates = {"cytokine_treatment": ("cytokine",)},
        perturbation_covariate_reps = {"cytokine_treatment": "esm2_embeddings"},
        sample_covariates = ["donor"],
        sample_covariate_reps = {"donor": "donor_embeddings"},
        split_covariates = ["donor"],
        max_combination_length = 1,
        null_value = 0.0,
    )

In [12]:
mem = memory_usage(run_cellflow1, max_usage=True)
print(f"Peak memory: {mem:.2f} MiB")

Peak memory: 163996.52 MiB


In [13]:
import os
import gc
import psutil
from memory_profiler import memory_usage

proc = psutil.Process(os.getpid())

def rss_mib() -> float:
    return proc.memory_info().rss / (1024**2)

def profile_call(name, fn, *args, interval=0.05, include_children=True, gc_collect=True, **kwargs):
    """
    Reports:
      - RSS before / after (MiB)
      - delta RSS
      - peak RSS during call (via memory_profiler sampling)
      - peak increase above start
    """
    if gc_collect:
        gc.collect()

    start_rss = rss_mib()

    # memory_usage returns MiB values over time; with retval=True we also get the function return.
    mem_series, ret = memory_usage(
        (fn, args, kwargs),
        interval=interval,
        max_usage=False,          # we want the series to compute peak and maybe debug
        retval=True,
        include_children=include_children,
    )

    end_rss = rss_mib()
    peak_rss = max(mem_series) if len(mem_series) else float("nan")

    print(
        f"{name:<20}  start={start_rss:10.2f} MiB  "
        f"end={end_rss:10.2f} MiB  "
        f"delta={end_rss-start_rss:10.2f} MiB  "
        f"peak={peak_rss:10.2f} MiB  "
        f"peak_inc={peak_rss-start_rss:10.2f} MiB"
    )
    return ret


In [14]:
import functools
import optax
from jax import nn

# 1) construction
cf = profile_call("CellFlow()", CellFlow, adata_train, solver="otfm")

# 2) prepare_data
profile_call(
    "prepare_data",
    cf.prepare_data,
    sample_rep="X_pca",
    control_key="is_control",
    perturbation_covariates={"cytokine_treatment": ("cytokine",)},
    perturbation_covariate_reps={"cytokine_treatment": "esm2_embeddings"},
    sample_covariates=["donor"],
    sample_covariate_reps={"donor": "donor_embeddings"},
    split_covariates=["donor"],
    max_combination_length=1,
    null_value=0.0,
)

layers_before_pool = {
    "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
    "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
}
layers_after_pool = {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0}
match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)

# 3) prepare_model
profile_call(
    "prepare_model",
    cf.prepare_model,
    condition_mode="deterministic",
    regularization=0.0,
    pooling="attention_token",
    pooling_kwargs={},
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.9,
    condition_encoder_kwargs={},
    pool_sample_covariates=True,
    time_freqs=1024,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="concatenation",
    decoder_dims=[4096, 4096, 4096],
    vf_act_fn=nn.silu,
    vf_kwargs=None,
    probability_path={"constant_noise": 0.5},
    match_fn=match_fn,
    optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
    solver_kwargs={},
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
)

# 4) train
profile_call(
    "train",
    cf.train,
    num_iterations=10,
    batch_size=1024,
    callbacks=[],
    valid_freq=20_000,
)


CellFlow()            start= 163931.54 MiB  end= 163931.54 MiB  delta=      0.00 MiB  peak= 196743.87 MiB  peak_inc=  32812.32 MiB
[########################################] | 100% Completed | 2.52 sms
[########################################] | 100% Completed | 208.97 ms
prepare_data          start= 163931.54 MiB  end= 165003.05 MiB  delta=   1071.51 MiB  peak= 200144.26 MiB  peak_inc=  36212.72 MiB


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


prepare_model         start= 165003.05 MiB  end= 165006.05 MiB  delta=      3.00 MiB  peak= 198885.74 MiB  peak_inc=  33882.68 MiB


100%|██████████| 10/10 [00:06<00:00,  1.66it/s]


train                 start= 165006.05 MiB  end= 165059.10 MiB  delta=     53.04 MiB  peak= 198996.20 MiB  peak_inc=  33990.15 MiB


In [2]:
#!/usr/bin/env python3
"""
profile_cellflow_cpu_memory.py

Profiles CPU RAM (RSS) step-by-step (peak + delta) for:
  1) preprocessing (pbmc_cytokines loading, obs columns, normalize/log, split, PCA)
  2) CellFlow pipeline (construct, prepare_data, prepare_model, train)

Outputs:
  - a pandas DataFrame printed to stdout
  - optional CSV written to --out_csv

Notes:
  - This is dependency-free (no memory_profiler); it samples process RSS via psutil.
  - It can include child processes (useful if dataloaders/spawned workers exist).
  - It uses vectorized ops instead of .apply for obs columns to reduce overhead.
"""

from __future__ import annotations

import argparse
import gc
import os
import threading
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

import pandas as pd
import psutil


# -------------------------
# Memory sampler utilities
# -------------------------
proc = psutil.Process(os.getpid())


def rss_mib(include_children: bool = True) -> float:
    """Resident set size (RSS) in MiB; optionally includes child processes."""
    rss = proc.memory_info().rss
    if include_children:
        try:
            for ch in proc.children(recursive=True):
                try:
                    rss += ch.memory_info().rss
                except psutil.Error:
                    pass
        except psutil.Error:
            pass
    return rss / (1024**2)


@dataclass
class StepRecord:
    step: str
    start_MiB: float
    end_MiB: float
    delta_MiB: float
    peak_MiB: float
    peak_inc_MiB: float
    wall_s: float


class MemoryProfiler:
    def __init__(self, interval_s: float = 0.02, include_children: bool = True, gc_collect: bool = True):
        self.interval_s = interval_s
        self.include_children = include_children
        self.gc_collect = gc_collect
        self.records: List[StepRecord] = []

    def profile(self, name: str, fn: Callable[..., Any], *args: Any, **kwargs: Any) -> Any:
        if self.gc_collect:
            gc.collect()

        start_rss = rss_mib(include_children=self.include_children)
        peak_rss = start_rss
        stop = {"flag": False}

        def sampler():
            nonlocal peak_rss
            while not stop["flag"]:
                m = rss_mib(include_children=self.include_children)
                if m > peak_rss:
                    peak_rss = m
                time.sleep(self.interval_s)

        t = threading.Thread(target=sampler, daemon=True)
        t0 = time.time()
        t.start()
        try:
            ret = fn(*args, **kwargs)
        finally:
            stop["flag"] = True
            t.join(timeout=1.0)
        t1 = time.time()

        end_rss = rss_mib(include_children=self.include_children)

        self.records.append(
            StepRecord(
                step=name,
                start_MiB=start_rss,
                end_MiB=end_rss,
                delta_MiB=end_rss - start_rss,
                peak_MiB=peak_rss,
                peak_inc_MiB=peak_rss - start_rss,
                wall_s=t1 - t0,
            )
        )
        return ret

    def to_df(self) -> pd.DataFrame:
        df = pd.DataFrame([r.__dict__ for r in self.records])
        for col in ["start_MiB", "end_MiB", "delta_MiB", "peak_MiB", "peak_inc_MiB"]:
            df[col.replace("MiB", "GiB")] = df[col] / 1024.0
        # Order columns
        cols = [
            "step",
            "wall_s",
            "start_MiB",
            "end_MiB",
            "delta_MiB",
            "peak_MiB",
            "peak_inc_MiB",
            "start_GiB",
            "end_GiB",
            "delta_GiB",
            "peak_GiB",
            "peak_inc_GiB",
        ]
        return df[cols]


# -------------------------
# Main experiment
# -------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--interval", type=float, default=0.02, help="RSS sampling interval in seconds (default: 0.02)")
    parser.add_argument(
        "--include-children",
        action="store_true",
        help="Include RSS of child processes (useful with dataloader workers)",
    )
    parser.add_argument(
        "--no-gc",
        action="store_true",
        help="Disable gc.collect() before each step (default: GC enabled)",
    )
    parser.add_argument("--out_csv", type=str, default="", help="Optional path to save CSV report")
    parser.add_argument("--num_iterations", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=1024)
    parser.add_argument("--valid_freq", type=int, default=20_000)
    args = parser.parse_args()

    mp = MemoryProfiler(
        interval_s=args.interval,
        include_children=args.include_children,
        gc_collect=(not args.no_gc),
    )

    # Imports inside main so that "spawn" start method (if you add it) can be set earlier if needed.
    import functools

    import scanpy as sc
    import optax
    from jax import nn

    import cellflow
    from cellflow import CellFlow
    import cellflow.preprocessing as cfpp
    from cellflow.matching import match_linear

    # -------------------------
    # Preprocessing
    # -------------------------
    adata = mp.profile("load pbmc_cytokines", cellflow.datasets.pbmc_cytokines)

    def add_obs_columns(a):
        a.obs["condition"] = a.obs["donor"].astype(str) + "_" + a.obs["cytokine"].astype(str)
        a.obs["is_control"] = a.obs["cytokine"].eq("PBS")
        return None

    mp.profile("add obs columns", add_obs_columns, adata)
    mp.profile("sc.pp.normalize_total", sc.pp.normalize_total, adata, target_sum=1e4)
    mp.profile("sc.pp.log1p", sc.pp.log1p, adata)

    def make_splits(a):
        adata_train = a[(a.obs["cytokine"] != "IL-15") | (a.obs["donor"] == "Donor8")].copy()
        adata_test = a[((a.obs["cytokine"] == "IL-15") & (a.obs["donor"] != "Donor8")) | (a.obs["cytokine"] == "PBS")].copy()
        return adata_train, adata_test

    adata_train, adata_test = mp.profile("split + copy train/test", make_splits, adata)

    mp.profile(
        "cfpp.centered_pca(train)",
        cfpp.centered_pca,
        adata_train,
        n_comps=100,
        method="rapids",
        keep_centered_data=False,
    )
    mp.profile("cfpp.project_pca(test<-train)", cfpp.project_pca, query_adata=adata_test, ref_adata=adata_train)

    # -------------------------
    # CellFlow execution
    # -------------------------
    cf = mp.profile("CellFlow()", CellFlow, adata_train, solver="otfm")

    mp.profile(
        "prepare_data",
        cf.prepare_data,
        sample_rep="X_pca",
        control_key="is_control",
        perturbation_covariates={"cytokine_treatment": ("cytokine",)},
        perturbation_covariate_reps={"cytokine_treatment": "esm2_embeddings"},
        sample_covariates=["donor"],
        sample_covariate_reps={"donor": "donor_embeddings"},
        split_covariates=["donor"],
        max_combination_length=1,
        null_value=0.0,
    )

    layers_before_pool = {
        "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
        "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
    }
    layers_after_pool = {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0}
    match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)

    mp.profile(
        "prepare_model",
        cf.prepare_model,
        condition_mode="deterministic",
        regularization=0.0,
        pooling="attention_token",
        pooling_kwargs={},
        layers_before_pool=layers_before_pool,
        layers_after_pool=layers_after_pool,
        condition_embedding_dim=256,
        cond_output_dropout=0.9,
        condition_encoder_kwargs={},
        pool_sample_covariates=True,
        time_freqs=1024,
        time_encoder_dims=[1024, 1024, 1024],
        time_encoder_dropout=0.0,
        hidden_dims=[2048, 2048, 2048],
        hidden_dropout=0.0,
        conditioning="concatenation",
        decoder_dims=[4096, 4096, 4096],
        vf_act_fn=nn.silu,
        vf_kwargs=None,
        probability_path={"constant_noise": 0.5},
        match_fn=match_fn,
        optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
        solver_kwargs={},
        layer_norm_before_concatenation=False,
        linear_projection_before_concatenation=False,
    )

    mp.profile(
        "train",
        cf.train,
        num_iterations=args.num_iterations,
        batch_size=args.batch_size,
        callbacks=[],
        valid_freq=args.valid_freq,
    )

    # -------------------------
    # Report
    # -------------------------
    df = mp.to_df()

    # Print nicely
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 140):
        print(df.to_string(index=False))

    if args.out_csv:
        df.to_csv(args.out_csv, index=False)
        print(f"\nSaved CSV report to: {args.out_csv}")



main()


usage: ipykernel_launcher.py [-h] [--interval INTERVAL] [--include-children]
                             [--no-gc] [--out_csv OUT_CSV]
                             [--num_iterations NUM_ITERATIONS]
                             [--batch_size BATCH_SIZE]
                             [--valid_freq VALID_FREQ]
ipykernel_launcher.py: error: unrecognized arguments: -f /ictstr01/home/icb/dominik.klein/.local/share/jupyter/runtime/kernel-c06013f2-eb75-48dd-ab12-1eb8aace0cb6.json


SystemExit: 2

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ============================================
# CPU RAM profiling (RSS) step-by-step (Notebook)
# ============================================
# Profiles both:
#  1) preprocessing (pbmc_cytokines, obs columns, normalize/log, split, PCA)
#  2) CellFlow execution (construct, prepare_data, prepare_model, train)
#
# Output:
#  - pandas DataFrame report (start/end/delta/peak/peak_inc + GiB conversions)
#
# Requirements:
#  - psutil, pandas
#  - scanpy, cellflow, cfpp, jax, optax (your usual env)
# ============================================

import os
import gc
import time
import threading
from dataclasses import dataclass
from typing import Any, Callable, List, Dict

import psutil
import pandas as pd

proc = psutil.Process(os.getpid())

def rss_mib(include_children: bool = True) -> float:
    """Resident set size (RSS) in MiB; optionally includes child processes."""
    rss = proc.memory_info().rss
    if include_children:
        try:
            for ch in proc.children(recursive=True):
                try:
                    rss += ch.memory_info().rss
                except psutil.Error:
                    pass
        except psutil.Error:
            pass
    return rss / (1024**2)

@dataclass
class StepRecord:
    step: str
    start_MiB: float
    end_MiB: float
    delta_MiB: float
    peak_MiB: float
    peak_inc_MiB: float
    wall_s: float

class NotebookMemoryProfiler:
    """
    Dependency-free peak RSS profiler that works well in notebooks.
    """
    def __init__(self, interval_s: float = 0.02, include_children: bool = True, gc_collect: bool = True):
        self.interval_s = interval_s
        self.include_children = include_children
        self.gc_collect = gc_collect
        self.records: List[StepRecord] = []

    def profile(self, name: str, fn: Callable[..., Any], *args: Any, **kwargs: Any) -> Any:
        if self.gc_collect:
            gc.collect()

        start_rss = rss_mib(include_children=self.include_children)
        peak_rss = start_rss
        stop = {"flag": False}

        def sampler():
            nonlocal peak_rss
            while not stop["flag"]:
                m = rss_mib(include_children=self.include_children)
                if m > peak_rss:
                    peak_rss = m
                time.sleep(self.interval_s)

        t = threading.Thread(target=sampler, daemon=True)
        t0 = time.time()
        t.start()
        try:
            ret = fn(*args, **kwargs)
        finally:
            stop["flag"] = True
            t.join(timeout=1.0)
        t1 = time.time()

        end_rss = rss_mib(include_children=self.include_children)

        self.records.append(
            StepRecord(
                step=name,
                start_MiB=start_rss,
                end_MiB=end_rss,
                delta_MiB=end_rss - start_rss,
                peak_MiB=peak_rss,
                peak_inc_MiB=peak_rss - start_rss,
                wall_s=t1 - t0,
            )
        )
        return ret

    def report(self) -> pd.DataFrame:
        df = pd.DataFrame([r.__dict__ for r in self.records])
        for col in ["start_MiB", "end_MiB", "delta_MiB", "peak_MiB", "peak_inc_MiB"]:
            df[col.replace("MiB", "GiB")] = df[col] / 1024.0

        cols = [
            "step", "wall_s",
            "start_MiB", "end_MiB", "delta_MiB", "peak_MiB", "peak_inc_MiB",
            "start_GiB", "end_GiB", "delta_GiB", "peak_GiB", "peak_inc_GiB",
        ]
        return df[cols]

# Instantiate profiler (tune interval to catch short spikes)
mp = NotebookMemoryProfiler(interval_s=0.01, include_children=True, gc_collect=True)


In [ ]:
# ============================================
# Imports for your pipeline
# ============================================

import functools
import scanpy as sc
import optax
from jax import nn

import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear


In [ ]:
# ============================================
# 1) Preprocessing (profiled)
# ============================================

# Load dataset
adata = mp.profile("load pbmc_cytokines", cellflow.datasets.pbmc_cytokines)

# Vectorized obs columns (semantically identical to your .apply)
def add_obs_columns(a):
    a.obs["condition"] = a.obs["donor"].astype(str) + "_" + a.obs["cytokine"].astype(str)
    a.obs["is_control"] = a.obs["cytokine"].eq("PBS")
    return None

mp.profile("add obs columns", add_obs_columns, adata)

# Normalize + log1p (in-place)
mp.profile("sc.pp.normalize_total", sc.pp.normalize_total, adata, target_sum=1e4)
mp.profile("sc.pp.log1p", sc.pp.log1p, adata)

# Split + copy
def make_splits(a):
    adata_train = a[(a.obs["cytokine"]!="IL-15") | (a.obs["donor"]=="Donor8")].copy()
    adata_test  = a[((a.obs["cytokine"]=="IL-15") & (a.obs["donor"]!="Donor8")) | (a.obs["cytokine"]=="PBS")].copy()
    return adata_train, adata_test

adata_train, adata_test = mp.profile("split + copy train/test", make_splits, adata)

# PCA train + projection to test
mp.profile(
    "cfpp.centered_pca(train)",
    cfpp.centered_pca,
    adata_train,
    n_comps=100,
    method="rapids",
    keep_centered_data=False,
)

mp.profile(
    "cfpp.project_pca(test<-train)",
    cfpp.project_pca,
    query_adata=adata_test,
    ref_adata=adata_train,
)

# (optional sanity check)
adata_train.n_obs, adata_test.n_obs


In [ ]:
# ============================================
# 2) CellFlow execution (profiled)
# ============================================

cf = mp.profile("CellFlow()", CellFlow, adata_train, solver="otfm")

mp.profile(
    "prepare_data",
    cf.prepare_data,
    sample_rep="X_pca",
    control_key="is_control",
    perturbation_covariates={"cytokine_treatment": ("cytokine",)},
    perturbation_covariate_reps={"cytokine_treatment": "esm2_embeddings"},
    sample_covariates=["donor"],
    sample_covariate_reps={"donor": "donor_embeddings"},
    split_covariates=["donor"],
    max_combination_length=1,
    null_value=0.0,
)

layers_before_pool = {
    "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
    "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
}
layers_after_pool = {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0}

match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)

mp.profile(
    "prepare_model",
    cf.prepare_model,
    condition_mode="deterministic",
    regularization=0.0,
    pooling="attention_token",
    pooling_kwargs={},
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.9,
    condition_encoder_kwargs={},
    pool_sample_covariates=True,
    time_freqs=1024,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="concatenation",
    decoder_dims=[4096, 4096, 4096],
    vf_act_fn=nn.silu,
    vf_kwargs=None,
    probability_path={"constant_noise": 0.5},
    match_fn=match_fn,
    optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
    solver_kwargs={},
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
)

mp.profile(
    "train",
    cf.train,
    num_iterations=10,
    batch_size=1024,
    callbacks=[],
    valid_freq=20_000,
)


In [ ]:
# ============================================
# 3) Report
# ============================================

df_mem = mp.report()

# Display in notebook
df_mem
